# Analysis.py

## imports

In [3]:
import pandas as pd
from pathlib import Path

In [4]:
dataset_name = "gold_student_exam_summary"
CURR_DIR = Path.cwd()
DATA_FILE = CURR_DIR.parent / "data" / "input" / "gold_student_exam_summary.csv"

In [5]:
df = pd.read_csv(DATA_FILE)

In [16]:
df.head()

,alias,exam_type,total_score,year,cohort,cohort_short,dob,gender,admitted_term,program_code,program_name,funding_source,study_mode,postcode,ethnicity,age_at_exam,age_bin
0,2016-15NW2580,unCKT,15,2016.0,2016.0,2016.0,15/07/1985,F,4153.0,3476.0,B. of Primary Education,CSS,FT,NaN,NABTRRS,30.0,30+
1,2016-15NW2580,CKT1,34,2016.0,2016.0,2016.0,15/07/1985,F,4153.0,3476.0,B. of Primary Education,CSS,FT,NaN,NABTRRS,30.0,30+
2,2016-14CR3225,unCKT,15,2016.0,2016.0,2016.0,10/07/1986,F,4157.0,3476.0,B. of Primary Education,CSS,FT,NaN,NABTRRS,29.0,25-29
3,2016-14CR3225,CKT1,24,2016.0,2016.0,2016.0,10/07/1986,F,4157.0,3476.0,B. of Primary Education,CSS,FT,NaN,NABTRRS,29.0,25-29
4,2016-13AC3542,unCKT,25,2016.0,2016.0,2016.0,30/04/1988,F,4133.0,3475.0,B.Education(Birth to Twelve),CSS,FT,NaN,NABTRRS,27.0,25-29


## Coding the analysis 

### 1. Converting example queries into pandas

- convert example queries to pandas to help develop functions
- e.g. age vs average unckt1
- using named_aggregation - df.agg(new_col_name=('original_col', 'sum'))
- phase 1: mvp, prototype the complete process for this analysis first all the way to display

In [7]:
df["exam_type"] == "unCKT"

0       True
1      False
2       True
3      False
4       True
       ...  
431    False
432    False
433     True
434    False
435     True
Name: exam_type, Length: 436, dtype: bool

In [66]:
# groupby gender and aggregate total_score, WHERE exam_type = unCKT
filtered_df = df[df["exam_type"] == "unCKT"]
filtered_df.groupby("gender").agg(avg_score=("total_score","mean"))

,avg_score
gender,
F,24.225434
M,27.527778


In [80]:
# groupby female and aggregate total_score, WHERE exam_type = unCKT and Gender = female
# TODO: how to make this more secure? not needed, this is gold data, that is a data cleaning issue 
filtered_df = df[(df["gender"]=="F") & (df["exam_type"]=="unCKT")]
filtered_df.groupby("gender").agg(avg_score=("total_score", "mean"))

,avg_score
gender,
F,24.225434


### 2. interpreting the filter from JSON

- it will need to take list containing each filter condition
- this could be for multiple columns
- possible solutions
    - isin() - this however is for multiple values against a single column
    - use np.logical_and to 'reduce' them into a single filter
    - Dynamic Filtering with .query()
- NB: in schemas.py -> FilterCondition class have defined operator: Literal["equals", "not_equals", "in"]


Handling Column-Specific Lists (Dictionary Mapping)
If you have a dictionary where keys are columns and values are lists of what to keep, you can loop through them to narrow down the DataFrame progressively.

In [35]:
# example analysis plan as received from LLM in JSON format
analysis_plan = {'dataset': 'gold_student_exam_summary', 
                 'metric_column': 'total_score', 
                 'aggregation': 'avg', 
                 'group_by': [], 
                  'filters': [{'column': 'exam_type', 'operator': 'equals', 'value': 'CKT1'}, 
                              {'column': 'gender', 'operator': 'equals', 'value': 'F'}, 
                              {'column': 'study_mode', 'operator': 'equals', 'value': 'FT'}], 
                 'clarification_required': False, 
                 'clarification_question': None}

In [36]:
# start with a dictionary
filters = analysis_plan["filters"]

NB: GPT recommends starting with basic mask = pd.Series(True, index=df.index) to include all rows and then add filters on top of that
- remember a mask e.g. df["exam_type"] == "unCKT", returns True/False.
- you then apply it to a data frame to return the rows where its true e.g. df[df["exam_type"] == "unCKT"]
- the raeson to start with a base mask = pd.Series(True, index=df.index) is that so if no filters are applied you still get the full table
- class pandas.Series(data=None, index=None, dtype=None, name=None, copy=None)

NB:
df[mask] usually returns a new filtered DataFrame object, and you should treat it as a copy, not a safe view.

In [23]:
def apply_filters(df, filters):
    mask = pd.Series(data=True, index=df.index)

    for f in filters:
        column = f["column"]
        value = f["value"]
        operator = f["operator"]

        if operator == "equals":
            condition = df[column] == value
        elif operator == "not_equals":
            condition = df[column] != value
        elif operator == "in":
            condition = df[column].isin(value)
        else:
            raise ValueError(f"Unsupported operator {operator}")

        mask = mask & condition
        
    
    return df[mask]

In [37]:
filtered_df = apply_filters(df, filters)
filtered_df.head()

,alias,exam_type,total_score,year,cohort,cohort_short,dob,gender,admitted_term,program_code,program_name,funding_source,study_mode,postcode,ethnicity,age_at_exam,age_bin
1,2016-15NW2580,CKT1,34,2016.0,2016.0,2016.0,15/07/1985,F,4153.0,3476.0,B. of Primary Education,CSS,FT,NaN,NABTRRS,30.0,30+
3,2016-14CR3225,CKT1,24,2016.0,2016.0,2016.0,10/07/1986,F,4157.0,3476.0,B. of Primary Education,CSS,FT,NaN,NABTRRS,29.0,25-29
5,2016-13AC3542,CKT1,37,2016.0,2016.0,2016.0,30/04/1988,F,4133.0,3475.0,B.Education(Birth to Twelve),CSS,FT,NaN,NABTRRS,27.0,25-29
7,2016-14ES3980,CKT1,54,2016.0,2016.0,2016.0,12/05/1988,F,4153.0,3476.0,B. of Primary Education,CSS,FT,2774.0,NABTRRS,27.0,25-29
8,2016-14SB4292,CKT1,43,2016.0,2016.0,2016.0,18/11/1989,F,4147.0,3476.0,B. of Primary Education,CSS,FT,NaN,NABTRRS,26.0,25-29


### 3. interpreting the aggregation
- e.g. converting filtered_df.groupby("gender").agg(avg_score=("total_score", "mean"))

In [42]:
#groupby gender then average score
filtered_df.groupby("gender").agg(avg_score=("total_score", "mean"))

,avg_score
gender,
F,39.410714


In [57]:
# aggregating without a group by
result = filtered_df["total_score"].agg("mean")
result

np.float64(39.410714285714285)

In [63]:
# multiple groupby
filtered_df.groupby(["ethnicity", "program_code"]).agg(avg_score=("total_score", "mean"))

avg_score
ethnicity program_code           
ABRGL     3477.0        38.000000
NABTRRS   3363.0        49.000000
          3475.0        37.481481
          3476.0        40.510204
          3477.0        40.338983
          3484.0        51.000000
NONE      3476.0        30.500000
          3477.0        33.000000

In [64]:
# multiple groupby alternate - this does not re-label the total_score to avg_score
filtered_df.groupby(["ethnicity", "program_code"])["total_score"].agg("mean").reset_index()

,ethnicity,program_code,total_score
0,ABRGL,3477.0,38.000000
1,NABTRRS,3363.0,49.000000
2,NABTRRS,3475.0,37.481481
3,NABTRRS,3476.0,40.510204
4,NABTRRS,3477.0,40.338983
5,NABTRRS,3484.0,51.000000
6,NONE,3476.0,30.500000
7,NONE,3477.0,33.000000


In [72]:
# analysis_plan with multiple group by
analysis_plan_multi_group = {'dataset': 'gold_student_exam_summary', 
                             'metric_column': 'total_score', 
                             'aggregation': 'mean', 
                             'group_by': ['ethnicity', 'program_name'], 
                             'filters': [{'column': 'gender', 'operator': 'equals', 'value': 'F'}, 
                                         {'column': 'exam_type', 'operator': 'equals', 'value': 'unCKT'}], 
                             'clarification_required': False, 
                             'clarification_question': None}

- when there's no group by, it will just return the aggregation result which is a single value
- therefore need to wrap this so it always returns a dataframe
- reset.index() fills out the index to include all the groupb categories (see above)

In [82]:
def apply_aggregation(filtered_df, metric_column, aggregation, group_by=None):
    """Applies the aggregation to the filtered dataframe
    
    """
    
    if group_by:
        return filtered_df.groupby(group_by, dropna=False).agg(avg_score=(metric_column,aggregation)).reset_index()
    else:
        value = filtered_df[metric_column].agg(aggregation)
        return pd.DataFrame({
            "metric": [metric_column],
            "aggregation": [aggregation],
            "value": [value]})
    

In [83]:
aggregated_df = apply_aggregation(filtered_df, "total_score", "mean", ['ethnicity', 'program_name'])
aggregated_df.head()

,ethnicity,program_name,avg_score
0,ABRGL,B.Prm Education(Religious Ed),26.000000
1,NABTRRS,B. of Primary Education,25.311111
2,NABTRRS,B.Comms and Media,5.000000
3,NABTRRS,B.Education Studies,27.000000
4,NABTRRS,B.Education(Birth to Twelve),22.836364


### 4. combining both into a single function

In [84]:
def execute_analysis(df, analysis_plan):
    ''' Applies the analysis plan (filtering, group by, and aggregation) to the dataframe
    '''
    # filter the df
    filters = analysis_plan["filters"]
    filtered_df = apply_filters(df, filters)

    #apply the groupby and aggregation
    metric_column = analysis_plan["metric_column"]
    aggregation = analysis_plan["aggregation"]
    group_by = analysis_plan["group_by"]
    
    return apply_aggregation(filtered_df, metric_column, aggregation, group_by)

    

In [86]:
final_df = execute_analysis(df, analysis_plan_multi_group)
final_df.head()

,ethnicity,program_name,avg_score
0,ABRGL,B.Prm Education(Religious Ed),26.000000
1,NABTRRS,B. of Primary Education,25.311111
2,NABTRRS,B.Comms and Media,5.000000
3,NABTRRS,B.Education Studies,27.000000
4,NABTRRS,B.Education(Birth to Twelve),22.836364
